
# Exercise 2
**Starting Point:** Solution from Exercise 1

---

## Add Quantum Processors

Begin by creating a quantum processor with 2 qubit positions. Apply a `DepolarNoiseModel` with a depolarization time of 100 ns to simulate qubit memory decoherence. Assign this processor to the `Alice` node during initialization.

Next, modify the `AliceProtocol` to store Alice’s qubit in the `QuantumProcessor` using the `put` method. After waiting for 30 ns, use the `peek` method to retrieve the qubit and measure it with the `measure` method of the `QuantumProcessor`. Verify that the simulation runs successfully.

To test noise effects, increase the waiting time at Alice to 1000 ns and observe that the density matrix and measurement results show reduced correlation. Finally, reset the waiting time back to 30 ns.

---

## Add Physical Instructions

Create `PhysicalInstruction` instances for qubit initialization, the Hadamard gate, and the CNOT gate. Configure the CNOT gate topology so that position 0 acts as the control and position 1 as the target. Set the duration of all operations to 10 ns.

Include these physical instructions during the initialization of the `QuantumProcessor`.

---

## Create a Bell State Using a QuantumProgram

Convert the circuit from the previous exercise into a `QuantumProgram`. Replace the `ns.qubits.create_qubits(2)` instruction by adding initialization instructions directly in the quantum program to create the qubits.

Develop the quantum program to generate a Bell state using H and CNOT gates. Verify that the simulation runs correctly and produces the expected results.


# Exercise code snippets
The following code snippets can be used to complete the exercise

In [ ]:
from netsquid.components import QuantumProcessor, QuantumProgram
from netsquid.components.models import FixedDelayModel, DepolarNoiseModel
from netsquid.components import PhysicalInstruction, INSTR_INIT, INSTR_H, INSTR_CNOT, INSTR_MEASURE, INSTR_CS
from netsquid.nodes import Node

# Components:
dummy_instruction = PhysicalInstruction(INSTR_CS, duration=0, topology=[(9,10)])
dummy_noise_model = DepolarNoiseModel(depolar_rate=0., time_independent=True)

quantum_processor = QuantumProcessor("Dummy memory", 
                        num_positions=100,
                        memory_noise_models=dummy_noise_model,
                        phys_instructions=[dummy_instruction])

# Create node with qmemory assigned
node = Node("Dummy node", qmemory=quantum_processor)

# Operations with quantum processor:
qubit, = ns.qubits.create_qubits(1)

node.qmemory.put(qubit, positions=5)
qubit, = node.qmemory.peek(positions=5) # Get qubit object, but don't take it out of the memory
qubit, = node.qmemory.pop(positions=5)  # Get qubit object, takes it out of the memory
node.qmemory.put(qubit, positions=1)
m, p = node.qmemory.measure(positions=1)

# Create a QuantumProgram
class DummyProgram(QuantumProgram):
    default_num_qubits = 2
    
    def program(self):
        q1, q2 = self.get_qubit_indices(2)
        self.apply(INSTR_CS, [q1, q2])
        yield self.run()

qprogram = DummyProgram()

# Execute quantum program
# NOTE: In a Program one should add a "yield" in order to wait for the QuantumProgram to finish execution
node.qmemory.execute_program(qprogram, qubit_mapping=[9,10])

# Exercise starting point:

In [ ]:
import netsquid as ns
ns.set_qstate_formalism(ns.QFormalism.DM)

## Network setup

In [ ]:
from netsquid.nodes import Node
from netsquid.components import Port, QuantumChannel
from netsquid.nodes import DirectConnection
from netsquid.components.models import FixedDelayModel, DepolarNoiseModel

alice_node = Node("Alice")
bob_node   = Node("Bob")

alice_port = Port("Alice_port", alice_node)
bob_port   = Port("Bob_port",   bob_node)

delay_model = FixedDelayModel(delay=10.)
error_model = DepolarNoiseModel(depolar_rate=0.2, time_independent=True)

channel_a_to_b = QuantumChannel("Channel Alice to Bob",
                                  models={"delay_model": delay_model,
                                          "quantum_noise_model": error_model}
                                  )

channel_b_to_a = QuantumChannel("Channel Bob to Alice",
                                  models={"delay_model": delay_model,
                                          "quantum_noise_model": error_model}
                                  )


connection = DirectConnection(name="connection",
                              channel_AtoB=channel_a_to_b,
                              channel_BtoA=channel_b_to_a)


alice_node.connect_to(remote_node=bob_node, connection=connection,
                     local_port_name="Alice_port", remote_port_name="Bob_port")

## Protocols

In [ ]:
from netsquid.protocols import NodeProtocol

class AliceProtocol(NodeProtocol):
    def run(self):
        port = self.node.ports["Alice_port"]

        q1, q2 = ns.qubits.create_qubits(2)

        ns.qubits.operate(q1, ns.H)
        ns.qubits.operate([q1, q2], ns.CNOT)
                
        port.tx_output(q2)
        print(f"{ns.sim_time()} ns Alice send q2")

        # Wait 30 ns
        yield self.await_timer(30)
        
        dm = ns.qubits.reduced_dm([q1, q2])
        print(f"{ns.sim_time()} ns Alice stops waiting, DM=\n{dm}")
        
        m, _ = ns.qubits.measure(q1)

        print(f"{ns.sim_time()} ns Alice measures q1: {m}")



class BobProtocol(NodeProtocol):
    def run(self):
        port = self.node.ports["Bob_port"]

        yield self.await_port_input(port)
        
        q2 = port.rx_input().items[0]

        dm = ns.qubits.reduced_dm(q2.qstate.qubits)
        print(f"{ns.sim_time()} ns Bob receives q2, DM=\n{dm}")

        m, _ = ns.qubits.measure(q2)

        print(f"{ns.sim_time()} ns Bob measures q2: {m}")

alice_protocol = AliceProtocol(node=alice_node)
bob_protocol = BobProtocol(node=bob_node)

## Run simulation

In [ ]:
# Start all protocols
alice_protocol.start()
bob_protocol.start()

sim_stats = ns.sim_run()

print(sim_stats)

# Reset simulation
alice_protocol.stop()
bob_protocol.stop()

ns.sim_reset()